# ISIC 2019 — FedProto Baseline

Extracts ISIC 2019 embeddings then runs FedProto.

**Dataset:** `andrewmvd/isic-2019`  |  **GPU T4**  |  **~90 min**

Save output as: `feduca-isic-fedproto-results`

In [ ]:
import os, json, time, copy, glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.utils.data import DataLoader, Dataset, TensorDataset
from PIL import Image
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import ttest_rel
import warnings; warnings.filterwarnings("ignore")

RESULTS_DIR = "/kaggle/working/results_isic_fedproto"
EMBED_DIR   = "/kaggle/working/isic2019_embeddings"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(EMBED_DIR,   exist_ok=True)

DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 8
EMBED_DIM   = 768
ALPHAS      = [1000, 1, 0.1]
SEEDS       = [0, 1, 2, 3, 4]
CLASS_NAMES = ["MEL","NV","BCC","AK","BKL","DF","VASC","SCC"]
K           = 4
print(f"Device: {DEVICE}")

gt_hits = glob.glob("/kaggle/input/**/ISIC_2019_Training_GroundTruth.csv",
                    recursive=True)
if not gt_hits:
    gt_hits = glob.glob("/kaggle/input/**/*GroundTruth*.csv", recursive=True)
if not gt_hits:
    raise FileNotFoundError("Add andrewmvd/isic-2019 dataset")

GT_CSV = gt_hits[0]
print(f"CSV: {GT_CSV}")
jpg_files  = glob.glob("/kaggle/input/**/*.jpg", recursive=True)
img_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in jpg_files}
print(f"Images indexed: {len(img_lookup)}")

In [ ]:
gt           = pd.read_csv(GT_CSV)
class_cols   = [c for c in gt.columns if c != "image"]
gt["label"]  = gt[class_cols].values.argmax(axis=1)
gt["lesion_id"] = gt["image"]

gss1 = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
tv_idx, te_idx = next(gss1.split(gt, gt["label"], groups=gt["lesion_id"]))
gt_tv   = gt.iloc[tv_idx].reset_index(drop=True)
gt_test = gt.iloc[te_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.111, random_state=42)
tr_idx, va_idx = next(gss2.split(gt_tv, gt_tv["label"], groups=gt_tv["lesion_id"]))
gt_train = gt_tv.iloc[tr_idx].reset_index(drop=True)
gt_val   = gt_tv.iloc[va_idx].reset_index(drop=True)

print(f"Train:{len(gt_train)} Val:{len(gt_val)} Test:{len(gt_test)}")
print(f"Classes: {class_cols}")

In [ ]:
print("Loading ViT-B/16 SWAG...")
weights_vit = ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1
backbone    = vit_b_16(weights=weights_vit).to(DEVICE)
backbone.eval()
for p in backbone.parameters(): p.requires_grad = False
print(f"Frozen: {sum(p.numel() for p in backbone.parameters()):,} params")

transform = T.Compose([
    T.Resize(384, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(384), T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

class ISICDataset(Dataset):
    def __init__(self, df, lookup, tfm):
        self.df=df.reset_index(drop=True); self.lookup=lookup; self.tfm=tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r=self.df.iloc[i]; p=self.lookup.get(r.image)
        img=Image.open(p).convert("RGB") if p else Image.new("RGB",(384,384))
        return self.tfm(img), int(r.label)

@torch.no_grad()
def extract(df, name, bs=64):
    ds=ISICDataset(df,img_lookup,transform)
    dl=DataLoader(ds,batch_size=bs,shuffle=False,num_workers=2,pin_memory=True)
    embs,lbls=[],[]
    t0=time.time()
    for bi,(imgs,labels) in enumerate(dl):
        x=backbone._process_input(imgs.to(DEVICE))
        n=x.shape[0]
        x=torch.cat([backbone.class_token.expand(n,-1,-1),x],dim=1)
        cls=backbone.encoder(x)[:,0,:].cpu().float().numpy()
        embs.append(cls); lbls.append(labels.numpy())
        done=min((bi+1)*bs,len(ds))
        if (bi+1)%30==0:
            ela=time.time()-t0
            print(f"  {name}: {done}/{len(ds)} | {ela:.0f}s | ETA {ela/done*(len(ds)-done):.0f}s")
    e=np.concatenate(embs); l=np.concatenate(lbls)
    print(f"  DONE {name}: {len(e)} embeddings | {time.time()-t0:.0f}s")
    return e,l

print("\nExtracting ISIC 2019 embeddings (train/val/test)...")
for name, df in [("train",gt_train),("val",gt_val),("test",gt_test)]:
    e,l=extract(df,name)
    np.savez_compressed(f"{EMBED_DIR}/embeddings_{name}.npz",embeddings=e,labels=l)
    print(f"  {name}: {e.shape}  classes:{dict(sorted(Counter(l.tolist()).items()))}")
print("\nEmbedding extraction complete.")

In [ ]:
def load_emb(name):
    d=np.load(f"{EMBED_DIR}/embeddings_{name}.npz",allow_pickle=True)
    return (torch.tensor(d["embeddings"],dtype=torch.float32),
            torch.tensor(d["labels"],dtype=torch.long))

emb_train,lbl_train=load_emb("train")
emb_val,  lbl_val  =load_emb("val")
emb_test, lbl_test =load_emb("test")
print(f"Train:{emb_train.shape} Val:{emb_val.shape} Test:{emb_test.shape}")

counts =torch.bincount(lbl_train,minlength=NUM_CLASSES).float()
weights=(1.0/counts); weights=(weights/weights.mean()).to(DEVICE)
print(f"Class weights: {weights.cpu().numpy().round(3)}")

class BottleneckMLP(nn.Module):
    def __init__(self,ratio=4,n_classes=8):
        super().__init__(); b=768//ratio
        self.net=nn.Sequential(
            nn.Linear(768,4*b),nn.GELU(),nn.LayerNorm(4*b),nn.Dropout(0.2),
            nn.Linear(4*b,b),  nn.GELU(),nn.LayerNorm(b),  nn.Dropout(0.2),
            nn.Linear(b,4*b),  nn.GELU(),nn.LayerNorm(4*b),nn.Dropout(0.2),
            nn.Linear(4*b,n_classes))
    def forward(self,x): return self.net(x)

@torch.no_grad()
def evaluate(head,embs,labels):
    head.eval(); p,t=[],[]
    for x,y in DataLoader(TensorDataset(embs,labels),batch_size=512):
        p.append(head(x.to(DEVICE)).argmax(1).cpu()); t.append(y)
    p=torch.cat(p).numpy(); t=torch.cat(t).numpy()
    f1s=[]
    for c in range(NUM_CLASSES):
        tp=((p==c)&(t==c)).sum(); fp=((p==c)&(t!=c)).sum(); fn=((p!=c)&(t==c)).sum()
        pr=tp/(tp+fp+1e-9); rc=tp/(tp+fn+1e-9); f1s.append(float(2*pr*rc/(pr+rc+1e-9)))
    return {"accuracy":float((p==t).mean()),"macro_f1":float(np.mean(f1s)),
            "f1_per_class":[float(x) for x in f1s]}

def dirichlet_partition(labels,K,alpha,seed):
    rng=np.random.default_rng(seed); idx=[[] for _ in range(K)]
    for c in range(NUM_CLASSES):
        ci=np.where(labels.numpy()==c)[0]; rng.shuffle(ci)
        props=rng.dirichlet(np.full(K,alpha))
        splits=(np.cumsum(props)*len(ci)).astype(int)[:-1]
        for k,chunk in enumerate(np.split(ci,splits)): idx[k].extend(chunk.tolist())
    return [np.array(i) for i in idx]

def fedavg_agg(states,sizes):
    total=sum(sizes)
    return {k:sum((sizes[i]/total)*states[i][k].float() for i in range(len(sizes)))
            for k in states[0]}

def compute_prototypes(embs,labels):
    proto=torch.zeros(NUM_CLASSES,EMBED_DIM); count=torch.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        mask=labels==c
        if mask.sum()>0: proto[c]=embs[mask].mean(0); count[c]=float(mask.sum())
    return proto,count

print("All utilities ready.")

In [ ]:
def local_train_fedproto(head,embs,labels,global_protos,
                          lr=5e-3,epochs=5,bs=32,lam=0.1):
    h=copy.deepcopy(head).to(DEVICE); h.train()
    opt=optim.Adam(h.parameters(),lr=lr)
    crit=nn.CrossEntropyLoss(weight=weights)
    local_protos={}
    for c in range(NUM_CLASSES):
        mask=(labels==c)
        if mask.sum()>0: local_protos[c]=embs[mask].mean(0).to(DEVICE)
    for _ in range(epochs):
        for x,y in DataLoader(TensorDataset(embs.to(DEVICE),labels.to(DEVICE)),
                               batch_size=bs,shuffle=True):
            opt.zero_grad()
            ce=crit(h(x),y)
            pl=torch.tensor(0.,device=DEVICE)
            for c,lp in local_protos.items():
                if c in global_protos:
                    pl+=torch.norm(lp-global_protos[c].to(DEVICE))**2
            (ce+lam*pl/NUM_CLASSES).backward(); opt.step()
    return copy.deepcopy(h.state_dict())

def fedproto_aggregate(states,sizes,client_protos):
    agg=fedavg_agg(states,sizes); global_protos={}
    for c in range(NUM_CLASSES):
        parts=[(client_protos[k][0][c],client_protos[k][1][c])
               for k in range(len(states)) if client_protos[k][1][c]>0]
        if parts:
            num=sum(p*n for p,n in parts); denom=sum(n for _,n in parts)
            global_protos[c]=num/denom
    return agg,global_protos

print("="*55)
print("FedProto on ISIC 2019 — K=4, 3 alpha x 5 seeds")
print("="*55)
fp_results=[]

for alpha in ALPHAS:
    for seed in SEEDS:
        partition  =dirichlet_partition(lbl_train,K,alpha,seed)
        c_embs     =[emb_train[idx] for idx in partition]
        c_labels   =[lbl_train[idx] for idx in partition]
        sizes      =[len(idx) for idx in partition]
        head       =BottleneckMLP(ratio=4,n_classes=NUM_CLASSES).to(DEVICE)
        global_protos={}; best_val=-1; best_round=0; best_state=None
        t0=time.time()

        for rnd in range(1,61):
            states=[]; client_protos=[]
            for k in range(K):
                st=local_train_fedproto(head,c_embs[k],c_labels[k],global_protos)
                states.append(st)
                client_protos.append(compute_prototypes(c_embs[k],c_labels[k]))
            agg_state,global_protos=fedproto_aggregate(states,sizes,client_protos)
            head.load_state_dict(agg_state)
            vm=evaluate(head,emb_val.to(DEVICE),lbl_val)
            if vm["macro_f1"]>best_val:
                best_val=vm["macro_f1"]; best_round=rnd
                best_state=copy.deepcopy(head.state_dict())

        head.load_state_dict(best_state)
        tm=evaluate(head,emb_test.to(DEVICE),lbl_test)
        res={"method":"fedproto","dataset":"ISIC2019",
             "alpha":alpha,"seed":seed,"best_round":best_round,
             "best_val_f1":best_val,"test":tm}
        fp_results.append(res)
        with open(os.path.join(RESULTS_DIR,f"fedproto_a{alpha}_s{seed}.json"),"w") as f:
            json.dump(res,f,indent=2)

    f1s=[r["test"]["macro_f1"] for r in fp_results if r["alpha"]==alpha]
    print(f"  FedProto alpha={alpha}: F1={np.mean(f1s):.3f}+/-{np.std(f1s):.3f}")

with open(os.path.join(RESULTS_DIR,"fedproto_isic_all.json"),"w") as f:
    json.dump(fp_results,f,indent=2)
print("\nAll FedProto results saved.")

In [ ]:
print("="*60)
print("COMPLETE ISIC 2019 TABLE")
print("="*60)

fedavg_isic={1000:(0.729,0.007),1:(0.696,0.027),0.1:(0.403,0.100)}
pga_isic   ={1000:(0.735,0.009),1:(0.688,0.016),0.1:(0.524,0.058)}
fp_isic    ={alpha:(np.mean([r["test"]["macro_f1"] for r in fp_results if r["alpha"]==alpha]),
                    np.std ([r["test"]["macro_f1"] for r in fp_results if r["alpha"]==alpha]))
             for alpha in ALPHAS}

print(f"\n{str():15} {str():>14} {str():>14} {str():>14}")
print(f"{'Method':<15} {'alpha=1000':>14} {'alpha=1':>14} {'alpha=0.1':>14}")
print("-"*59)
for name,res in [("FedAvg",fedavg_isic),("FedProto",fp_isic),("PGA (Ours)",pga_isic)]:
    row=[f"{v[0]:.3f}+/-{v[1]:.3f}" for v in [res[a] for a in ALPHAS]]
    print(f"{name:<15} {row[0]:>14} {row[1]:>14} {row[2]:>14}")

fa=fedavg_isic[0.1][0]; fp=fp_isic[0.1][0]; pg=pga_isic[0.1][0]
print(f"\nKey finding at alpha=0.1:")
print(f"  FedProto: {fp:.3f}  vs FedAvg: {fp-fa:+.3f}")
print(f"  PGA     : {pg:.3f}  vs FedAvg: {pg-fa:+.3f}")
print(f"  PGA vs FedProto: {pg-fp:+.3f}")
print(f"\nFedProto is WORSE than FedAvg on ISIC 2019 (same as HAM10000)")
print(f"PGA advantage confirmed on second dataset")
print(f"\nSave output as: feduca-isic-fedproto-results")
print(f"Then share the results here for paper update.")